# 03 — La couche agentique : outil -> MCP -> agent ReAct

Au-dessus du pipeline RAG fixe, trois niveaux d'outillage :

1. **RAG-comme-outil** (`tools/rag_tool.py`) — un contrat standard (nom, description, schéma JSON).
2. **Serveur MCP** (`rag_mcp_server.py`) — expose l'outil via le Model Context Protocol.
3. **Agent ReAct** (`core/agent.py`) — un LLM qui raisonne et décide *quand* chercher (Pensée -> Action -> Observation).

Le LLM n'émet qu'une intention ; c'est le code validé (`run_tool`) qui exécute.

> Prérequis : Ollama + MongoDB démarrés, corpus ingéré. La cellule de l'agent prend ~1 min (8B).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

## 1) RAG-comme-outil — le contrat

Un nom, une description (sur laquelle un LLM se fonde pour décider quand l'employer), un schéma JSON.

In [2]:
from tools.rag_tool import tool_spec, run_tool
spec = tool_spec()
print('Outil :', spec['name'])
print('Description :', spec['description'][:120], '…')
print('Schéma params :', json.dumps(spec['parameters']['properties'], ensure_ascii=False)[:160], '…')

# Exécution validée (mode passages = récupération seule, rapide).
res = run_tool('rag_search', {'query': 'niveau EAL de la TOE', 'mode': 'passages', 'max_passages': 4})
print(f"\nrun_tool → ok={res['ok']} hors_scope={res['hors_scope']} passages={len(res.get('passages', []))}")

Outil : rag_search
Description : Recherche dans la base documentaire technique (cibles de sécurité ANSSI / Critères Communs) et renvoie une réponse sourc …
Schéma params : {"query": {"type": "string", "description": "La question, en langage naturel."}, "document": {"type": "string", "description": "Nom exact du document pour restr …


C:\Users\laury\Desktop\rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


21:11:42 INFO    rag.rerank | [rerank] Chargement Cross-Encoder local: C:\Users\laury\Desktop\rag_project\models\bge-reranker-v2-m3 (device=cuda:0)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8094.61it/s]


run_tool → ok=True hors_scope=False passages=4


## 2) Serveur MCP — l'outil annoncé via le protocole

Le serveur enregistre l'outil ; un hôte MCP (ex. Claude Desktop) le découvre via `list_tools`.
(En notebook, on utilise le top-level `await` : le kernel a déjà une boucle asyncio.)

In [3]:
import rag_mcp_server as srv
tools = await srv.mcp.list_tools()
for t in tools:
    print('Outil MCP :', t.name)
    print('  params :', list((t.inputSchema or {}).get('properties', {})))

Outil MCP : rag_search
  params : ['query', 'document']


## 3) Agent ReAct (streamé)

L'agent décide *lui-même* d'appeler `rag_search`, récupère des passages, puis **synthétise**
une réponse unique sourcée. On consomme le flux d'événements (vitesse perçue type Claude/ChatGPT).

In [4]:
from core.agent import ReActAgent

answer, sources, ntok = '', [], 0
for ev in ReActAgent().run_stream('Quel est le niveau EAL de la TOE ?'):
    k = ev['type']
    if k == 'thought':
        print('[Pensée]   ', ev['text'][:90])
    elif k == 'action':
        print('[Recherche]', ev['input'].get('query', ''))
    elif k == 'observation':
        print('[Obs]      ', ev['text'])
    elif k == 'answer_token':
        ntok += 1
    elif k == 'done':
        r = ev['result']
        answer, sources = r['answer'], r['sources']
        print(f"\n[OK] {r['tool_calls']} recherche(s), {ntok} tokens streamés, {r['latency_s']}s ({r['stopped_reason']})")

print('\n--- REPONSE ---\n', answer[:400])
print('\n--- SOURCES ---')
for s in sources[:5]:
    print(f"  [{s.get('idx')}] {s.get('source')}")

[Pensée]    Je cherche à connaître le niveau EAL de la TOE, il me semble que cela pourrait être mentio
[Recherche] niveau EAL de la TOE


[Obs]       6 passage(s) trouvé(s)


[Pensée]    Je cherche à connaître le niveau EAL de la TOE, il me semble que cela pourrait être mentio
[Recherche] niveau EAL de la TOE
[Obs]       (recherche déjà effectuée)



[OK] 1 recherche(s), 305 tokens streamés, 72.08s (synthesized)

--- REPONSE ---
 [Réponse]
Le nivel d'évaluation approprie pour l'objectif X correspond à un niveaueal augmenter avec des critères spécifique. Selon les informations fournies dans [1] et[2], nous pouvons identifier le niveau EAL de la TOE comme étant :

*   Niveau : 3 (élevée)
    *      Augmente par ALC\_FLR\.3
        AVA\_\_VLA.\
            ADV-LLD.
1, 
2,
 etADVIMP.

[Justification]
"Le niveau EAL de la TOE e

--- SOURCES ---
  [1] ANSSI-CC-cible_2011-1-20.md
  [2] ANSSI-CC-cible_2011-1-20.md
  [3] ANSSI-CC-cible_2011-1-20.md
  [4] ANSSI-CC-cible_2011-1-20.md
  [5] ANSSI-CC-cible_2011-1-20.md


## À retenir

- Le **même contrat d'outil** sert au tool calling, au MCP et à l'agent — pas de duplication.
- L'agent fait du **retrieval agentique** (il décide des recherches) puis **une seule génération**
  finale — au lieu d'une génération jetée à chaque recherche.
- Le **streaming** rend l'attente fluide même quand le total reste élevé (limite matérielle 8 Go ;
  cf. README → la vraie accélération est l'infra ou un modèle hébergé).